# Projet Machine Learning

### Groupe 3

**Membres :**
- PONCHON Léo
- MOUFOK Sabrina
- HUANG Lei
- ZHANG Yaqi
- AYDEMIR Yagmur

## 0. Préparation

In [ ]:
# Librairies
import pandas as pd
import numpy as np
import re
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import VotingClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split, cross_val_score, KFold, GridSearchCV
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score, precision_score, recall_score
from imblearn.over_sampling import SMOTE
from sklearn.feature_selection import chi2, SelectKBest

import plotly.graph_objs as go #graphiques avancés plotly
import plotly.offline as py #mode offline plotly
import plotly.express as px #visualisation rapide
import plotly.io as pio #sauvegarde d'image

# Librairies NLTK
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer #lemmatisation

from nltk import sent_tokenize #decoupage en phrases
from nltk import word_tokenize #decoupage en mots
from nltk import pos_tag #etiquetage grammatical
from nltk.stem.snowball import SnowballStemmer # Stemmatisation
from nltk.stem import PorterStemmer
from nltk.corpus import stopwords #stopwords

# Téléchargement des ressources NLTK
nltk.download('stopwords', quiet=True) #stopwords
nltk.download('wordnet', quiet=True) #WordNet
nltk.download('omw-1.4', quiet=True)


nltk.download("punkt") #tokenisation
nltk.download("punkt_tab") #tokenisation
nltk.download("averaged_perceptron_tagger") # Tags
nltk.download("averaged_perceptron_tagger_eng") # Tags anglais
nltk.download("tagsets_json") #Liste de tags
nltk.download("tagsets") #Ancienne liste

#import contractions
from collections import Counter # Comptage d'occurrences
from wordcloud import WordCloud # Generation de nuages de mots


In [ ]:
# Chargement du dataset
df = pd.read_csv("./scitweets_export.tsv", sep="\t")

## 1. Comprendre les données

### 1.1 Aperçu global

In [ ]:
# Vérifier les colonnes
# Comprendre les colonnes
print(df.columns)
print(df.head())

In [ ]:
# Vérifier les données
print(df["science_related"].value_counts())

print(df[["scientific_claim",
    "scientific_reference",
    "scientific_context"]].astype(int).sum())

In [ ]:
# Incohérences
incoherence1 = df[(df["science_related"] == 0) & ((df["scientific_claim"] == 1) | (df["scientific_reference"] == 1) | (df["scientific_context"] == 1))]
print(f"Incohérence 1: {len(incoherence1)}")

incoherence2 = df[(df["science_related"] == 1) & ((df["scientific_claim"] == 0) & (df["scientific_reference"] == 0) & (df["scientific_context"] == 0))]
print(f"Incohérence 2: {len(incoherence2)}")

In [ ]:
# data frame de science related
df_sci = df[df["science_related"] == 1]

# situation single-label
# {CLAIM}
claim = df_sci[(df_sci["scientific_claim"] == 1) & (df_sci["scientific_reference"] == 0) & (df_sci["scientific_context"] == 0)]

# {REF}
ref = df_sci[(df_sci["scientific_claim"] == 0) & (df_sci["scientific_reference"] == 1) & (df_sci["scientific_context"] == 0)]

# {CONTEXT}
context = df_sci[(df_sci["scientific_claim"] == 0) & (df_sci["scientific_reference"] == 0) & (df_sci["scientific_context"] == 1)]

# situation multi-label
# {CLAIM + REF}
claim_ref = df_sci[(df_sci["scientific_claim"] == 1) & (df_sci["scientific_reference"] == 1) & (df_sci["scientific_context"] == 0)]

# {CLAIM + CONTEXT}
claim_context = df_sci[(df_sci["scientific_claim"] == 1) & (df_sci["scientific_reference"] == 0) & (df_sci["scientific_context"] == 1)]

# {REF + CONTEXT}
ref_context = df_sci[(df_sci["scientific_claim"] == 0) & (df_sci["scientific_reference"] == 1) & (df_sci["scientific_context"] == 1)]

# {CLAIM + REF + CONTEXT}
all_three = df_sci[(df_sci["scientific_claim"] == 1) & (df_sci["scientific_reference"] == 1) & (df_sci["scientific_context"] == 1)]


print(f"Only Claim: {len(claim)}")
print(f"Only Reference: {len(ref)}")
print(f"Only Context: {len(context)}")
print(f"Claim + Ref: {len(claim_ref)}")
print(f"Claim + Context: {len(claim_context)}")
print(f"Ref + Context: {len(ref_context)}")
print(f"Claim + Ref + Context: {len(all_three)}")

print("-------------------------")
# task2 multi label
m_task2 = len(claim_context) + len(ref_context) + len(all_three)
# task multi label
m_task3 = len(claim_ref) + len(claim_context) + len(ref_context) + len(all_three)

print(f"Multi labels pour le task 2: {m_task2}")
print(f"Multi labels pour le task 3: {m_task3}")

### 1.2 Visualisation la distribution des labels

In [ ]:
# Vérifier l’équilibrage
df["science_related"].value_counts().plot(kind="bar")
plt.show()


In [ ]:
df["scientific_claim"].value_counts().plot(kind="bar")
plt.show()

In [ ]:
df["scientific_reference"].value_counts().plot(kind="bar")
plt.show()

In [ ]:
df["scientific_context"].value_counts().plot(kind="bar")
plt.show()

In [ ]:
df_sci["scientific_claim"].value_counts().plot(kind="bar")
plt.show()

In [ ]:
df_sci["scientific_reference"].value_counts().plot(kind="bar")
plt.show()

In [ ]:
df_sci["scientific_context"].value_counts().plot(kind="bar")
plt.show()

In [ ]:
label_statistique = {
    "Science Related": len(df_sci),
    "Only Claim": len(claim),
    "Only Ref": len(ref),
    "Only Context": len(context),
    "Claim + Ref": len(claim_ref),
    "Claim + Context": len(claim_context),
    "Ref + Context": len(ref_context),
    "All Three": len(all_three)
}

s = pd.Series(label_statistique)

plt.figure(figsize=(12, 6))
ax = s.plot(kind="bar")

plt.title("Distribution des label Combinatoires ", fontsize=14)
plt.xlabel("Combination de label", fontsize=12)
plt.ylabel("Nombre", fontsize=12)

for i, v in enumerate(s):
    ax.text(i, v + 2, str(v), ha='center', fontweight='bold')

plt.xticks(rotation=45, ha='right')
plt.show()

### 1.3 Les word clouds

In [ ]:
#Une première analyse des documents
#les Word Clouds
#sans nettoyage

# Garder seulement les textes non vides
texts = df["text"].dropna().astype(str)

# Concaténer tous les tweets en un seul gros texte
corpus = " ".join(texts)

# Découpage très simple
words = corpus.split()

# Comptage
word_counts = Counter(words)

# Word cloud
wc = WordCloud(width=1000, height=500, background_color="white")
wc.generate_from_frequencies(word_counts)

plt.figure(figsize=(14, 7))
plt.imshow(wc, interpolation="bilinear")
plt.axis("off")
plt.show()

# wc.to_file("./outputs/1-comprendre_les_données/worldcloud.png")

## 2. Prétraitements

### 2.1 Les founctions pour nettoyer les textes

In [ ]:
# Initialisation du lemmatizer
lemmatizer = WordNetLemmatizer()

# Initialisation du stemmer
stemmer = PorterStemmer()

# stop words
stop_words = set(stopwords.words("english"))
custom_stopwords = {"rt", "amp", "http", "via", "im", "dont", "cant", "u", "ur", "re", "ve", "ll", "s"}
final_stop_words = stop_words.union(custom_stopwords)

# Dictionnaire pour transformer emojis en mots
emoji_dict = {
    '🏀': ' basketball ', '💪': ' strong ', '✋': ' stop ', '💃': ' dance ',
    '💖': ' love ', '💀': ' dead ', '😉': ' wink ', '😷': ' sick ',
    '😧': ' surprised ', '😐': ' neutral ', '☀': ' sunny ', '😎': ' cool ',
    '❤': ' love ', '😊': ' happy ', '😬': ' nervous ', '💃🏻': ' dancing ',
    '😜': ' playful ', '🤦🏽': ' facepalm ', '♀': ' female ', '🤟': ' love ',
    '💸': ' money ', '👀': ' eyes ', '😈': ' devil ', '💔': ' broken ',
    '💙': ' love ', '🌊': ' wave ', '♻': ' recycle ', '👊': ' fist ',
    '👇': ' down ', '🌍': ' earth ', '💕': ' love ', '😄': ' happy ',
    '♥': ' love ', '👧': ' girl ', '💗': ' love ', '🐦': ' bird ',
    '🤣': ' laughing ', '🙄': ' eye roll ', '☑': ' checked ', '☞': ' point ',
    '❌': ' wrong ', '🎥': ' movie ', '📲': ' phone ', '👏': ' clap ',
    '🌞': ' sun ', '😃': ' happy ', '🙆🏻': ' okay ', '♂': ' male ',
    '🙋': ' wave ', '🦁': ' lion ', '💛': ' love ', '🍹': ' drink ',
    '🍷': ' wine ', '🤬': ' angry ', '📸': ' camera ', '☹': ' sad ',
    '👌🏿': ' okay ', '👌🏻': ' okay ', '👌🏽': ' okay ', '👌🏼': ' okay ',
    '👌🏾': ' okay ', '🧡': ' love ', '🗽': ' statue ', '🌼': ' flower ',
    '🌱': ' plant ', '🌸': ' flower ', '🌺': ' flower ', '👁': ' eye ',
    '✨': ' sparkle ', '😍': ' love ', '💥': ' boom ', '✌🏼': ' victory ',
    '🙏🏼': ' pray ', '✈': ' plane ', '🙂': ' smile ', '🤢': ' sick ',
    '😅': ' nervous ', '✧': ' star ', '🍃': ' leaf ', '💐': ' bouquet ',
    '🍁': ' leaf ', '👉': ' point ', '➡': ' right ', '➔': ' right ',
    '💯': ' hundred ', '✔': ' check ', '➙': ' arrow ', '😢': ' sad ',
    '😡': ' angry '
}

def clean_tweet(text):
    for e in emoji_dict:                          # remplacer les emojis par texte
        text = text.replace(e, emoji_dict[e])
    text = str(text).lower()                      # mettre tout en minuscule
    text = re.sub(r"http\S+|www\S+", " ", text)   # enlèver les URLs
    text = re.sub(r"@\w+", " ", text)             # enlèver les mentions
    text = re.sub(r"#", "", text)                 # garder le mot du hashtag
    text = re.sub(r"[^a-z\s]", " ", text)         # garder que les lettres
    text = re.sub(r"\s+", " ", text).strip()      # enlèver les espaces
    return text

def tokenize_text(text):
    return word_tokenize(text)

def remove_stopwords(tokens):
    return [w for w in tokens if w.lower() not in final_stop_words]

def lemmatize_tokens(tokens):
    return [lemmatizer.lemmatize(w) for w in tokens]

def join_tokens(tokens):
    return " ".join(tokens)

def stem_tokens(tokens):
    return [stemmer.stem(w) for w in tokens]

### 2.2 les Versions différentes de nettoyge

In [ ]:
# plusieurs représentations du même tweet

# V1 = nettoyage simple
df["text_clean_v1"] = df["text"].apply(clean_tweet)

# V2 = nettoyage + tokenisation + suppression stopwords
df["tokens_v2"] = df["text_clean_v1"].apply(tokenize_text)
df["tokens_v2"] = df["tokens_v2"].apply(remove_stopwords)
df["text_clean_v2"] = df["tokens_v2"].apply(join_tokens)

# V3 = nettoyage + tokenisation + stopwords + lemmatisation
df["tokens_v3"] = df["text_clean_v1"].apply(tokenize_text)
df["tokens_v3"] = df["tokens_v3"].apply(remove_stopwords)
df["tokens_v3"] = df["tokens_v3"].apply(lemmatize_tokens)
df["text_clean_v3"] = df["tokens_v3"].apply(join_tokens)

# V4 = nettoyage + tokenisation + stopwords + stemmer
df["tokens_v4"] = df["text_clean_v1"].apply(tokenize_text)
df["tokens_v4"] = df["tokens_v4"].apply(remove_stopwords)
df["tokens_v4"] = df["tokens_v4"].apply(stem_tokens)
df["text_clean_v4"] = df["tokens_v4"].apply(join_tokens)

### 2.3 Les résultats de nettoyage

In [ ]:
# comparaison
df[["text", "text_clean_v1", "text_clean_v2", "text_clean_v3", "text_clean_v4"]].head(5)

In [ ]:
# word cloud pour comparaison
plt.figure(figsize=(18,6))

for i, col in enumerate(["text_clean_v1", "text_clean_v2", "text_clean_v3"]):
    plt.subplot(1,3,i+1)

    corpus = " ".join(df[col].dropna().astype(str))
    wc = WordCloud(width=800, height=400, background_color="white", stopwords=[]).generate(corpus)

    plt.imshow(wc)
    plt.title(col)
    plt.axis("off")

plt.show()

### 2.4 Définir les 3 tâches par les textes nottoyées

### 2.4 Règler le problem de multi-lables

In [ ]:

# Les colonnes science_related, scientific_claim, scientific_reference existent déjà dans le dataset

# CONTEXT = tweets science-related qui ne sont ni claim ni reference
df["scientific_context"] = ((df["science_related"] == 1) &
                            (df["scientific_claim"] == 0) &
                            (df["scientific_reference"] == 0)).astype(int)

# Première tache, SCI vs. NON-SCI
task1 = df[df["science_related"].isin([0, 1])].copy()
task1["label"] = task1["science_related"].astype(int)

# Deuxième tache, CLAIM, REF vs. {CONTEXT}
# On combine CLAIM et REF comme une seule classe
task2 = df[(df["science_related"] == 1)].copy()
# Label: 1 = CLAIM ou REF, 0 = CONTEXT
task2["label"] = ((task2["scientific_claim"] == 1) | (task2["scientific_reference"] == 1)).astype(int)

# Dernière tache, {CLAIM} vs. {REF} vs. {CONTEXT}
# On crée une seule colonne de label avec 3 classes
task3 = df[(df["science_related"] == 1)].copy()

def create_3class_label(row):
    if row["scientific_claim"] == 1:
        return 0
    elif row["scientific_reference"] == 1:
        return 1
    elif row["scientific_context"] == 1:
        return 2
    else:
        return -1

task3["label"] = task3.apply(create_3class_label, axis=1)

# On garde uniquement les labels valides
task3 = task3[task3["label"].isin([0, 1, 2])].copy()

print("DISTRIBUTION DES CLASSES")
print(f"Tâche 1 (SCI vs NON-SCI): {len(task1)} samples")
print(task1["label"].value_counts())

print(f"Tâche 2 (CLAIM/REF vs CONTEXT): {len(task2)} samples")
print(task2["label"].value_counts())

print(f"Tâche 3 (CLAIM vs REF vs CONTEXT): {len(task3)} samples")
print(task3["label"].value_counts())

In [ ]:
# Visualisation de la distribution des classes
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Tache 1
task1['label'].value_counts().plot(kind='bar', ax=axes[0], color=['red', 'green'])
axes[0].set_title('Tâche 1: SCI vs NON-SCI')
axes[0].set_xlabel('Classe (0=NON-SCI, 1=SCI)')
axes[0].set_ylabel('Nombre')

# Tache 2
task2['label'].value_counts().plot(kind='bar', ax=axes[1], color=['blue', 'orange'])
axes[1].set_title('Tâche 2: CLAIM/REF vs CONTEXT')
axes[1].set_xlabel('Classe (0=CONTEXT, 1=CLAIM/REF)')
axes[1].set_ylabel('Nombre')

# Tache 3
task3['label'].value_counts().sort_index().plot(kind='bar', ax=axes[2], color=['red', 'blue', 'green'])
axes[2].set_title('Tâche 3: CLAIM vs REF vs CONTEXT')
axes[2].set_xlabel('Classe (0=CLAIM, 1=REF, 2=CONTEXT)')
axes[2].set_ylabel('Nombre')
axes[2].set_xticklabels(['CLAIM', 'REF', 'CONTEXT'], rotation=0)

plt.tight_layout()
plt.savefig('./outputs/class_distribution.png', dpi=150)
plt.show()

# Calcul du déséquilibre
print("\nRATIO DE DÉSÉQUILIBRE")
for name, task in [('Tâche 1', task1), ('Tâche 2', task2), ('Tâche 3', task3)]:
    counts = task['label'].value_counts()
    ratio = counts.max() / counts.min()
    print(f"{name}: {ratio:.2f}:1 (max/min)")

In [ ]:
# Définition du pattern pour détecter les emojis dans le texte (aide de l'IA uniquement pour trouver les codes UTF des emojis)
emoji_pattern = re.compile(
    "[\U0001F600-\U0001F64F"
    "\U0001F300-\U0001F5FF"
    "\U0001F680-\U0001F6FF"
    "\U0001F700-\U0001F77F"
    "\U0001F780-\U0001F7FF"
    "\U0001F800-\U0001F8FF"
    "\U0001F900-\U0001F9FF"
    "\U0001FA00-\U0001FAFF"
    "\U00002700-\U000027BF"
    "\U00002600-\U000026FF"
    "]+",
    re.UNICODE
)

def extract_emojis_from_df(df):
    emojis = []

    # On parcourt chaque colonne de df
    for colonne in df.columns:
        # Puis chaque cellule de la colonne
        for valeur in df[colonne]:
            texte = str(valeur)

            # On cherche les emojis dans le texte
            emojis_trouves = emoji_pattern.findall(texte)

            # On les ajoute à la liste globale
            emojis.extend(emojis_trouves)

    return emojis


# Extraction
emojis = extract_emojis_from_df(df)

# Suppression des doublons en mettant dans un ensemble, puis tri
emojis_uniques = sorted(set(emojis))

# Affichage
print("\nEmojis uniques :", emojis_uniques)

In [ ]:
# FONCTIONS DE PRÉTRAITEMENT

# Initialisation des outils NLP
lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words('english'))

# Ajout de mots fréquents dans les tweets qu'on veut ignorer
tweet_stop_words = ['rt', 'amp', 'via', 'us', 'im', 'dont', 'cant',
                    'u', 'ur', 're', 've', 'll', 's']

for word in tweet_stop_words:
    stop_words.add(word)

# Dictionnaire pour transformer emojis en mots
emoji_dict = {
    '🏀': ' basketball ', '💪': ' strong ', '✋': ' stop ', '💃': ' dance ',
    '💖': ' love ', '💀': ' dead ', '😉': ' wink ', '😷': ' sick ',
    '😧': ' surprised ', '😐': ' neutral ', '☀': ' sunny ', '😎': ' cool ',
    '❤': ' love ', '😊': ' happy ', '😬': ' nervous ', '💃🏻': ' dancing ',
    '😜': ' playful ', '🤦🏽': ' facepalm ', '♀': ' female ', '🤟': ' love ',
    '💸': ' money ', '👀': ' eyes ', '😈': ' devil ', '💔': ' broken ',
    '💙': ' love ', '🌊': ' wave ', '♻': ' recycle ', '👊': ' fist ',
    '👇': ' down ', '🌍': ' earth ', '💕': ' love ', '😄': ' happy ',
    '♥': ' love ', '👧': ' girl ', '💗': ' love ', '🐦': ' bird ',
    '🤣': ' laughing ', '🙄': ' eye roll ', '☑': ' checked ', '☞': ' point ',
    '❌': ' wrong ', '🎥': ' movie ', '📲': ' phone ', '👏': ' clap ',
    '🌞': ' sun ', '😃': ' happy ', '🙆🏻': ' okay ', '♂': ' male ',
    '🙋': ' wave ', '🦁': ' lion ', '💛': ' love ', '🍹': ' drink ',
    '🍷': ' wine ', '🤬': ' angry ', '📸': ' camera ', '☹': ' sad ',
    '👌🏿': ' okay ', '👌🏻': ' okay ', '👌🏽': ' okay ', '👌🏼': ' okay ',
    '👌🏾': ' okay ', '🧡': ' love ', '🗽': ' statue ', '🌼': ' flower ',
    '🌱': ' plant ', '🌸': ' flower ', '🌺': ' flower ', '👁': ' eye ',
    '✨': ' sparkle ', '😍': ' love ', '💥': ' boom ', '✌🏼': ' victory ',
    '🙏🏼': ' pray ', '✈': ' plane ', '🙂': ' smile ', '🤢': ' sick ',
    '😅': ' nervous ', '✧': ' star ', '🍃': ' leaf ', '💐': ' bouquet ',
    '🍁': ' leaf ', '👉': ' point ', '➡': ' right ', '➔': ' right ',
    '💯': ' hundred ', '✔': ' check ', '➙': ' arrow ', '😢': ' sad ',
    '😡': ' angry '
}

# FONCTION DE NETTOYAGE
def clean_text(text):
    # on met tout en minuscule
    text = str(text).lower()

    # on enlève les liens
    text = re.sub(r'http\S+|www\.\S+', '', text)

    # on enlève les @mentions
    text = re.sub(r'@\w+', '', text)

    # on enlève juste le # des hashtags
    text = re.sub(r'#', '', text)

    # on remplace les emojis par leur équivalent en texte
    for e in emoji_dict:
        text = text.replace(e, emoji_dict[e])

    # on garde que les lettres (le reste -> espace)
    text = re.sub(r'[^a-zA-Z\s]', ' ', text)

    # on enlève les espaces en trop
    text = re.sub(r'\s+', ' ', text).strip()

    # on découpe en mots
    words = text.split()

    # on enlève les stopwords et mots qui font pas plus de 2 caractères, puis on lemmatise
    res = []
    for w in words:
        if w not in stop_words and len(w) > 2:
            res.append(lemmatizer.lemmatize(w))

    return " ".join(res)


# APPLICATION AUX DATASETS
task1['text_clean'] = task1['text'].apply(clean_text)
task2['text_clean'] = task2['text'].apply(clean_text)
task3['text_clean'] = task3['text'].apply(clean_text)


# STATISTIQUES
print("\nLONGUEUR DES TEXTES NETTOYÉS")

for nom, dataframe in [
    ('Tâche 1', task1),
    ('Tâche 2', task2),
    ('Tâche 3', task3)
]:
    moyenne_mots = dataframe['text_clean'].apply(
        lambda txt: len(txt.split())
    ).mean()

    print(f"{nom} : {moyenne_mots} mots en moyenne")

In [ ]:
# FONCTIONS UTILITAIRES POUR LA MODÉLISATION
def prepare_task_data(task_df, test_size=0.2, random_state=42):
    X = task_df['text_clean']
    y = task_df['label']

    # Split
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=test_size, random_state=random_state, stratify=y
    )

    # Vectorisation TF-IDF avec n-grammes
    vec = TfidfVectorizer(
        ngram_range=(1, 2),  # Unigrams + bigrams
        max_features=5000,
        min_df=2,
        max_df=0.95
    )
    X_train_v = vec.fit_transform(X_train)
    X_test_v = vec.transform(X_test)

    smote = SMOTE(random_state=random_state)
    X_train_v, y_train = smote.fit_resample(X_train_v, y_train)

    return X_train_v, X_test_v, y_train, y_test, vec


def evaluate_with_cv(X_train_v, y_train, classifier, cv=10):
    kfold = KFold(n_splits=cv, shuffle=True, random_state=42)
    scores = cross_val_score(classifier, X_train_v, y_train, cv=kfold, scoring='accuracy')
    return scores.mean(), scores.std()


def train_and_evaluate(X_train_v, X_test_v, y_train, y_test, classifier, name):
    clf = classifier.fit(X_train_v, y_train)
    y_pred = clf.predict(X_test_v)

    acc = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred, average='weighted')
    precision = precision_score(y_test, y_pred, average='weighted')
    recall = recall_score(y_test, y_pred, average='weighted')

    return {
        'name': name,
        'accuracy': acc,
        'f1': f1,
        'precision': precision,
        'recall': recall,
        'y_pred': y_pred,
        'model': clf
    }


def plot_confusion_matrix(y_test, y_pred, title, labels=None):
    cm = confusion_matrix(y_test, y_pred)
    plt.figure(figsize=(6, 5))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=labels, yticklabels=labels)
    plt.title(f'Matrice de Confusion - {title}')
    plt.xlabel('Prédit')
    plt.ylabel('Réel')
    plt.tight_layout()
    plt.savefig(f'./outputs/cm_{title.replace(" ", "_")}.png', dpi=150)
    plt.show()

print("OK Fonctions de modélisation définies")

In [ ]:
#
# TÂCHE 1: SCI vs NON-SCI
#


print("TÂCHE 1: SCI vs NON-SCI")


# Préparation des données AVEC SMOTE
X_train_v, X_test_v, y_train, y_test, vec = prepare_task_data(task1)

print(f"Training set: {X_train_v.shape}")
print(f"Test set: {X_test_v.shape}")
print(f"Training labels distribution: {np.bincount(y_train)}")
# hmm la distribution est vraiment déséquilibrée...

# Définir les classifiers
classifiers = {
    'Naive Bayes': MultinomialNB(),
    'SVM': LinearSVC(max_iter=5000, random_state=42),
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Decision Tree': DecisionTreeClassifier(random_state=42),
    'KNN': KNeighborsClassifier(n_neighbors=5),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
}

# Évaluer chaque classifier
results_task1 = []

for name, clf in classifiers.items():
    # Validation croisée
    cv_mean, cv_std = evaluate_with_cv(X_train_v, y_train, clf)

    # Entraînement et test
    result = train_and_evaluate(X_train_v, X_test_v, y_train, y_test, clf, name)
    result['cv_mean'] = cv_mean
    result['cv_std'] = cv_std
    results_task1.append(result)

    print(f"\n{name}:")
    print(f"  CV Accuracy: {cv_mean:.4f} (+/- {cv_std:.4f})")
    print(f"  Test Accuracy: {result['accuracy']:.4f}")
    print(f"  F1-Score: {result['f1']:.4f}")

In [ ]:
# Tableau comparatif Tâche 1
results_df1 = pd.DataFrame([{
    'Classifier': r['name'],
    'CV Accuracy': r['cv_mean'],
    'CV Std': r['cv_std'],
    'Test Accuracy': r['accuracy'],
    'F1-Score': r['f1'],
    'Precision': r['precision'],
    'Recall': r['recall']
} for r in results_task1])

print("\nRÉSULTATS COMPARATIFS - TÂCHE 1")
print(results_df1.sort_values('F1-Score', ascending=False).to_string(index=False))

# Matrice de confusion pour le meilleur modèle
best_task1 = max(results_task1, key=lambda x: x['f1'])
print(f"\nMEILLEUR MODÈLE TÂCHE 1: {best_task1['name']}")
print(classification_report(y_test, best_task1['y_pred'], target_names=['NON-SCI', 'SCI']))
plot_confusion_matrix(y_test, best_task1['y_pred'], 'Task1_Best', ['NON-SCI', 'SCI'])

In [ ]:
#
# TÂCHE 2: CLAIM/REF vs CONTEXT
#


print("TÂCHE 2: CLAIM/REF vs CONTEXT")


# Préparation des données AVEC SMOTE
X_train_v, X_test_v, y_train, y_test, vec = prepare_task_data(task2)

print(f"Training set: {X_train_v.shape}")
print(f"Test set: {X_test_v.shape}")
print(f"Training labels distribution: {np.bincount(y_train)}")
# hmm la distribution est vraiment déséquilibrée...

# Évaluer chaque classifier
results_task2 = []

for name, clf in classifiers.items():
    # Validation croisée
    cv_mean, cv_std = evaluate_with_cv(X_train_v, y_train, clf)

    # Entraînement et test
    result = train_and_evaluate(X_train_v, X_test_v, y_train, y_test, clf, name)
    result['cv_mean'] = cv_mean
    result['cv_std'] = cv_std
    results_task2.append(result)

    print(f"\n{name}:")
    print(f"  CV Accuracy: {cv_mean:.4f} (+/- {cv_std:.4f})")
    print(f"  Test Accuracy: {result['accuracy']:.4f}")
    print(f"  F1-Score: {result['f1']:.4f}")

In [ ]:
# Tableau comparatif Tâche 2
results_df2 = pd.DataFrame([{
    'Classifier': r['name'],
    'CV Accuracy': r['cv_mean'],
    'CV Std': r['cv_std'],
    'Test Accuracy': r['accuracy'],
    'F1-Score': r['f1'],
    'Precision': r['precision'],
    'Recall': r['recall']
} for r in results_task2])

print("\nRÉSULTATS COMPARATIFS - TÂCHE 2")
print(results_df2.sort_values('F1-Score', ascending=False).to_string(index=False))

# Matrice de confusion pour le meilleur modèle
best_task2 = max(results_task2, key=lambda x: x['f1'])
print(f"\nMEILLEUR MODÈLE TÂCHE 2: {best_task2['name']}")
print(classification_report(y_test, best_task2['y_pred'], target_names=['CONTEXT', 'CLAIM/REF']))
plot_confusion_matrix(y_test, best_task2['y_pred'], 'Task2_Best', ['CONTEXT', 'CLAIM/REF'])

In [ ]:
#
# TÂCHE 3: CLAIM vs REF vs CONTEXT (3-classes)
#


print("TÂCHE 3: CLAIM vs REF vs CONTEXT")


# Préparation des données AVEC SMOTE
X_train_v, X_test_v, y_train, y_test, vec = prepare_task_data(task3)

print(f"Training set: {X_train_v.shape}")
print(f"Test set: {X_test_v.shape}")
print(f"Training labels distribution: {np.bincount(y_train)}")
# hmm la distribution est vraiment déséquilibrée...

# Évaluer chaque classifier
results_task3 = []

for name, clf in classifiers.items():
    # Validation croisée
    cv_mean, cv_std = evaluate_with_cv(X_train_v, y_train, clf)

    # Entraînement et test
    result = train_and_evaluate(X_train_v, X_test_v, y_train, y_test, clf, name)
    result['cv_mean'] = cv_mean
    result['cv_std'] = cv_std
    results_task3.append(result)

    print(f"\n{name}:")
    print(f"  CV Accuracy: {cv_mean:.4f} (+/- {cv_std:.4f})")
    print(f"  Test Accuracy: {result['accuracy']:.4f}")
    print(f"  F1-Score: {result['f1']:.4f}")

In [ ]:
# Tableau comparatif Tâche 3
results_df3 = pd.DataFrame([{
    'Classifier': r['name'],
    'CV Accuracy': r['cv_mean'],
    'CV Std': r['cv_std'],
    'Test Accuracy': r['accuracy'],
    'F1-Score': r['f1'],
    'Precision': r['precision'],
    'Recall': r['recall']
} for r in results_task3])

print("\nRÉSULTATS COMPARATIFS - TÂCHE 3")
print(results_df3.sort_values('F1-Score', ascending=False).to_string(index=False))

# Matrice de confusion pour le meilleur modèle
best_task3 = max(results_task3, key=lambda x: x['f1'])
print(f"\nMEILLEUR MODÈLE TÂCHE 3: {best_task3['name']}")
print(classification_report(y_test, best_task3['y_pred'], target_names=['CLAIM', 'REF', 'CONTEXT']))
plot_confusion_matrix(y_test, best_task3['y_pred'], 'Task3_Best', ['CLAIM', 'REF', 'CONTEXT'])

In [ ]:
# OPTIMISATION DES HYPERPARAMÈTRES AVEC GRIDSEARCHCV

# Grilles de paramètres pour les meilleurs classifiers
param_grids = {
    'Logistic Regression': {
        'C': [0.1, 1, 10],
        'max_iter': [1000]
    },
    'SVM': {
        'C': [0.1, 1, 10],
        'max_iter': [5000]
    },
    'Random Forest': {
        'n_estimators': [50, 100],
        'max_depth': [5, 10, None]
    }
}

# Optimisation pour chaque tâche

print("OPTIMISATION DES HYPERPARAMÈTRES")


best_params_results = {}

for task_name, task_data, results_list in [
    ('Tâche 1', task1, results_task1),
    ('Tâche 2', task2, results_task2),
    ('Tâche 3', task3, results_task3)
]:
    print(f"\n--- {task_name} ---")

    # Préparer les données
    X_train_v, X_test_v, y_train, y_test, vec = prepare_task_data(task_data)

    # Trouver le meilleur classifier de base
    best_clf_name = max(results_list, key=lambda x: x['f1'])['name']
    print(f"Meilleur classifier de base: {best_clf_name}")

    # Optimiser avec GridSearchCV
    if best_clf_name in param_grids:
        base_clf = classifiers[best_clf_name]

        grid_search = GridSearchCV(
            base_clf,
            param_grids[best_clf_name],
            cv=5,
            scoring='f1_weighted',
            n_jobs=-1
        )

        grid_search.fit(X_train_v, y_train)

        print(f"Meilleurs paramètres: {grid_search.best_params_}")
        print(f"Meilleur score CV: {grid_search.best_score_:.4f}")

        # Évaluer sur le test set
        y_pred = grid_search.best_estimator_.predict(X_test_v)
        test_f1 = f1_score(y_test, y_pred, average='weighted')
        test_acc = accuracy_score(y_test, y_pred)

        print(f"Test Accuracy: {test_acc:.4f}")
        print(f"Test F1-Score: {test_f1:.4f}")

        best_params_results[task_name] = {
            'classifier': best_clf_name,
            'params': grid_search.best_params_,
            'cv_score': grid_search.best_score_,
            'test_f1': test_f1,
            'test_acc': test_acc
        }

In [ ]:
#
# SÉLECTION DE FEATURES AVEC CHI² (BONUS)
#


print("SÉLECTION DE FEATURES (BONUS)")


def get_top_features(task_df, task_name, top_k=20):
    # Préparer les données (sans SMOTE pour garder la distribution originale)
    X = task_df['text_clean']
    y = task_df['label']

    # Vectorisation
    vec = TfidfVectorizer(ngram_range=(1, 2), max_features=5000)
    X_v = vec.fit_transform(X)

    # Chi² test
    chi2_scores, p_values = chi2(X_v, y)

    # Créer un DataFrame des features
    feature_names = vec.get_feature_names_out()
    feature_scores = pd.DataFrame({
        'feature': feature_names,
        'chi2_score': chi2_scores
    }).sort_values('chi2_score', ascending=False)

    print(f"\nTop {top_k} features - {task_name}")
    print(feature_scores.head(top_k).to_string(index=False))

    return feature_scores.head(top_k)['feature'].tolist()

# Extraire les features pour chaque tâche
features_task1 = get_top_features(task1, 'Tâche 1: SCI vs NON-SCI')
features_task2 = get_top_features(task2, 'Tâche 2: CLAIM/REF vs CONTEXT')
features_task3 = get_top_features(task3, 'Tâche 3: CLAIM vs REF vs CONTEXT')

In [ ]:
# TABLEAU COMPARATIF FINAL

print("TABLEAU COMPARATIF FINAL")


# Créer un tableau comparatif pour chaque tâche
def create_comparison_table(results_list, task_name):
    df = pd.DataFrame([{
        'Task': task_name,
        'Classifier': r['name'],
        'CV Accuracy': f"{r['cv_mean']:.4f} ± {r['cv_std']:.4f}",
        'Test Accuracy': f"{r['accuracy']:.4f}",
        'F1-Score': f"{r['f1']:.4f}"
    } for r in results_list])
    return df

comparison_df = pd.concat([
    create_comparison_table(results_task1, 'Task 1: SCI vs NON-SCI'),
    create_comparison_table(results_task2, 'Task 2: CLAIM/REF vs CONTEXT'),
    create_comparison_table(results_task3, 'Task 3: CLAIM vs REF vs CONTEXT')
])

print(comparison_df.to_string(index=False))

# Sauvegarder le tableau
comparison_df.to_csv('./outputs/comparison_results.csv', index=False)
print("\nOK Résultats sauvegardés dans outputs/comparison_results.csv")

# Sources utiles
Le TP1 a fourni les bases de l'exploration et de la visualisation des données. Ensuite, on a utilisé le TP2 pour comprendre les outils de classification, validation croisée et optimisation.
Enfin, le projet gitlab a fourni le traitement spécifique aux données textuelles (TF-IDF, NLTK, prétraitement).
Pour trouver tout les emojis dans le tsv, une fonction python externe a été développée pour les récupérer.